# Throughput: GB/s of f32 against the number of workers

`bench_throughput.zig` times `bufferToWorker` with 1 to 12 workers, at sizes from 16,384 to 262,144 blocks per call, with the plate size and threshold of `main.zig`.
The figure keeps the best working spot: the size where the throughput is highest. Throughput counts the f32 bytes written (64 per block).
Line: median; band: interquartile range over the repetitions (20). The y axis is capped at 50 GB/s, so the band is clipped at 6 and 7 workers.

In [ ]:
import re
import subprocess
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# the notebook sits in Claude Benchmark/<name>/, the repository two levels up
HERE = Path.cwd().resolve()
ROOT = HERE.parents[1]

MM = 1 / 25.4
INK, PAPER, GREY, SHADE = "#1f1d1a", "#faf8f3", "#77746e", "#d3cfc6"
BASELINE_GREY = "#a3a09a"
plt.rcParams.update({
    "font.family": "Liberation Sans", "font.size": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.spines.top": False, "axes.spines.right": False,
})


def main_constant(name):
    """A constant of src/main.zig, such as `const MIN_BLOCKS_FOR_WORKERS = 16 * 1024;`, evaluated."""
    source = (ROOT / "src" / "main.zig").read_text(encoding="utf-8")
    expression = re.search(rf"^const {name} = ([0-9 *]+);", source, re.MULTILINE).group(1)
    return int(np.prod([int(factor) for factor in expression.split("*")]))


def run_benchmark(bench, csv):
    """Runs the benchmark (calls of bufferToWorker only) when its results are missing."""
    if not (HERE / csv).exists():
        subprocess.run(["zig", "run", "-OReleaseFast", "-mcpu=native", "--dep", "dequantizer",
                        f"-Mroot={HERE / bench}", f"-Mdequantizer={ROOT / 'src' / 'root.zig'}"],
                       cwd=ROOT, check=True)
    return pd.read_csv(HERE / csv)


In [ ]:
data = run_benchmark("bench_throughput.zig", "throughput.csv")
cpus = int(data["cpus"].iloc[0])
data["gbps"] = data["blocks"] * 64 / data["ns_per_call"]
summary = data.groupby(["blocks", "workers"])["gbps"].quantile([0.25, 0.5, 0.75]).unstack()
best_size = summary[0.5].idxmax()[0]
spot = summary.loc[best_size]
print(f"best working spot: {best_size:,} blocks per call ({best_size * 64 / 2**20:.0f} MB of f32), peak {spot[0.5].max():.1f} GB/s")
summary[0.5].unstack().round(1)

In [ ]:
colour = matplotlib.colormaps["magma"](0.30)  # purple
workers = spot.index.to_numpy()
fig, ax = plt.subplots(figsize=(140 * MM, 90 * MM))
fig.subplots_adjust(left=0.10, right=0.95, top=0.93, bottom=0.14)
ax.fill_between(workers, spot[0.25], spot[0.75], color=colour, alpha=0.2, lw=0)
ax.plot(workers, spot[0.5], color=colour, lw=1.2, marker="o", ms=3, markeredgewidth=0)

ax.annotate(f"{spot[0.5].iloc[0]:.1f} GB/s", (1, spot[0.5].iloc[0]), xytext=(8, -6), textcoords="offset points",
            ha="left", va="top", fontsize=6.5, fontweight="bold")

# beyond the physical cores, two threads share one core's vector units
cores = cpus // 2
ax.axvline(cores + 0.5, color=INK, lw=0.8, ls=":", zorder=0)
ax.text(cores + 0.5, 0.03, "← 1 thread per core   ", transform=ax.get_xaxis_transform(), color=INK, fontsize=6.5, ha="right")
ax.text(cores + 0.5, 0.03, "   2 per core →", transform=ax.get_xaxis_transform(), color=INK, fontsize=6.5, ha="left")
# past the cores the medians form a plateau whose bands overlap: label the range, not one peak
plateau = spot.loc[cores:, 0.5]
ax.text(workers[-1], 30, f"plateau {plateau.min():.0f}–{plateau.max():.0f} GB/s", ha="right", va="top",
        fontsize=6.5, fontweight="bold")

ax.set_xticks(workers)
ax.set_xlabel("Workers", labelpad=8)

ax.set_ylabel("Throughput (GB/s of f32)", labelpad=8)
ax.set_ylim(0, 50)
fig.savefig(HERE / "throughput.png", dpi=300)